# 02. 특성 공학 (Feature Engineering) - Colab 버전

## 목적
재료 Tokenizer 구축 및 학습 데이터셋 생성

## 핵심 구현
1. IngredientTokenizer: BERT 스타일 토크나이저
2. DynamicMaskingDataset: 동적 마스킹 (20%~80%)
3. NegativeSamplingDataset: Contrastive Learning

## 입력
- `data/processed/recipe_v3/recipes_normalized.pkl`

## 출력
- `data/processed/recipe_v3/tokenizer.json`
- `data/processed/recipe_v3/dataset_info.pt`

In [2]:
# [1단계] Colab 환경 설정 및 Drive 마운트
import sys
import os

# Colab 환경 감지
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # 프로젝트 루트 설정 (본인의 Drive 경로에 맞게 수정)
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone'
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, f'{PROJECT_ROOT}/notebooks')

    print(f"✅ Colab 환경 감지")
    print(f"📁 프로젝트 루트: {PROJECT_ROOT}")
else:
    from pathlib import Path
    PROJECT_ROOT = Path().resolve().parent
    sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))

    print("💻 로컬 환경에서 실행 중")
    print(f"📁 프로젝트 루트: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Colab 환경 감지
📁 프로젝트 루트: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone


In [3]:
# [2단계] 라이브러리 임포트
import gc
import json
import random
from pathlib import Path
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Colab에서는 문자열을 Path로 변환
if IN_COLAB:
    PROJECT_ROOT = Path(PROJECT_ROOT)

# Gap Filling 유틸리티 임포트
from utils.gap_filling import (
    IngredientTokenizer,
    DynamicMaskingDataset,
    NegativeSamplingDataset,
    LeaveOneOutDataset,
    collate_fn
)

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

프로젝트 루트: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone
PyTorch 버전: 2.9.0+cu126
CUDA 사용 가능: True
GPU: NVIDIA L4


In [4]:
# 경로 설정
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'recipe_v3'
OUTPUT_DIR = DATA_DIR

# 데이터 로드 (Pickle 형식)
df = pd.read_pickle(DATA_DIR / 'recipes_normalized.pkl')
recipes = df['ingredients'].tolist()

print(f"레시피 수: {len(recipes):,}")
print(f"샘플: {recipes[0][:5]}")

레시피 수: 202,181
샘플: ['떡국떡\x07\x07g\x07', '소고기\x07\x07g\x07', '멸치육수\x07\x07ml\x07', '파', '계란\x07\x07개\x07']


## 1. Tokenizer 구축

In [5]:
# 토크나이저 초기화 및 어휘 구축
tokenizer = IngredientTokenizer()

print("어휘 사전 구축 중...")
tokenizer.build_vocab(
    recipes,
    min_freq=5,        # 최소 5회 이상 등장
    max_vocab_size=None  # 제한 없음
)

print(f"\n=== Tokenizer 정보 ===")
print(f"어휘 크기: {tokenizer.vocab_size:,}")
print(f"재료 수 (특수 토큰 제외): {tokenizer.num_ingredients:,}")

어휘 사전 구축 중...

=== Tokenizer 정보 ===
어휘 크기: 12,089
재료 수 (특수 토큰 제외): 12,084


In [6]:
# 특수 토큰 확인
print("\n=== 특수 토큰 ===")
print(f"[PAD] ID: {tokenizer.PAD_ID}")
print(f"[MASK] ID: {tokenizer.MASK_ID}")
print(f"[UNK] ID: {tokenizer.UNK_ID}")
print(f"[CLS] ID: {tokenizer.CLS_ID}")
print(f"[SEP] ID: {tokenizer.SEP_ID}")


=== 특수 토큰 ===
[PAD] ID: 0
[MASK] ID: 1
[UNK] ID: 2
[CLS] ID: 3
[SEP] ID: 4


In [7]:
# 가장 빈번한 재료 확인
print("\n=== 상위 20개 재료 ===")
for ing, freq in tokenizer.get_most_common(20):
    token_id = tokenizer.get_token_id(ing)
    print(f"  {ing} (ID={token_id}): {freq:,}회")


=== 상위 20개 재료 ===
  파 (ID=5): 66,587회
  마늘 (ID=6): 61,389회
  양파 (ID=7): 51,695회
  간장 (ID=8): 51,490회
  설탕 (ID=9): 47,337회
  고추가루 (ID=10): 45,558회
  고추 (ID=11): 45,287회
  소금 (ID=12): 45,021회
  계란 (ID=13): 39,053회
  후추 (ID=14): 38,218회
  참기름 (ID=15): 35,576회
  생크림 (ID=16): 29,227회
  버섯 (ID=17): 22,584회
  식용유 (ID=18): 20,751회
  당근 (ID=19): 20,474회
  통깨 (ID=20): 20,186회
  생선 (ID=21): 18,731회
  조개 (ID=22): 18,001회
  돼지고기 (ID=23): 16,652회
  감자 (ID=24): 15,753회


In [8]:
# 인코딩 테스트
test_recipe = ['돼지고기', '양파', '마늘', '간장', '설탕']
print(f"\n원본: {test_recipe}")

encoded = tokenizer.encode(test_recipe, max_len=16, return_attention_mask=True)
print(f"인코딩: {encoded['input_ids']}")
print(f"어텐션 마스크: {encoded['attention_mask']}")

decoded = tokenizer.decode(encoded['input_ids'])
print(f"디코딩: {decoded}")


원본: ['돼지고기', '양파', '마늘', '간장', '설탕']
인코딩: [3, 23, 7, 6, 8, 9, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0]
어텐션 마스크: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
디코딩: ['돼지고기', '양파', '마늘', '간장', '설탕']


In [9]:
# 토크나이저 저장
tokenizer_path = OUTPUT_DIR / 'tokenizer.json'
tokenizer.save(str(tokenizer_path))
print(f"토크나이저 저장: {tokenizer_path}")

토크나이저 저장: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/tokenizer.json


## 2. 레시피 인코딩

In [10]:
# 모든 레시피 인코딩 (특수 토큰 제외, 재료 ID만)
def encode_recipes_simple(recipes: List[List[str]], tokenizer) -> List[List[int]]:
    """레시피를 재료 ID 리스트로 변환 (특수 토큰 미포함)"""
    encoded = []
    for recipe in tqdm(recipes, desc="인코딩"):
        ids = [
            tokenizer.get_token_id(ing)
            for ing in recipe
        ]
        # UNK 토큰 필터링
        ids = [i for i in ids if i != tokenizer.UNK_ID]
        if len(ids) >= 2:  # 최소 2개 재료
            encoded.append(ids)
    return encoded

encoded_recipes = encode_recipes_simple(recipes, tokenizer)
print(f"\n인코딩된 레시피 수: {len(encoded_recipes):,}")

인코딩:   0%|          | 0/202181 [00:00<?, ?it/s]


인코딩된 레시피 수: 202,153


In [11]:
# 인코딩 결과 확인
print("\n=== 인코딩 샘플 ===")
for i in range(3):
    original = recipes[i]
    encoded = encoded_recipes[i]
    decoded = tokenizer.decode(encoded)
    print(f"\n[{i}] 원본: {original[:5]}...")
    print(f"    인코딩: {encoded[:5]}...")
    print(f"    디코딩: {decoded[:5]}...")


=== 인코딩 샘플 ===

[0] 원본: ['떡국떡\x07\x07g\x07', '소고기\x07\x07g\x07', '멸치육수\x07\x07ml\x07', '파', '계란\x07\x07개\x07']...
    인코딩: [1078, 384, 1209, 5, 80]...
    디코딩: ['떡국떡\x07\x07g\x07', '소고기\x07\x07g\x07', '멸치육수\x07\x07ml\x07', '파', '계란\x07\x07개\x07']...

[1] 원본: ['돼지고기', '된장\x07\x07큰술\x07']...
    인코딩: [23, 437]...
    디코딩: ['돼지고기', '된장\x07\x07큰술\x07']...

[2] 원본: ['돼지고기', '양파\x07\x07개\x07', '감자', '파', '배추']...
    인코딩: [23, 54, 24, 5, 29]...
    디코딩: ['돼지고기', '양파\x07\x07개\x07', '감자', '파', '배추']...


## 3. 학습/검증 분할

In [12]:
# 학습/검증 분할 (90:10)
TRAIN_RATIO = 0.9

n_total = len(encoded_recipes)
n_train = int(n_total * TRAIN_RATIO)
n_val = n_total - n_train

# 셔플 인덱스
indices = list(range(n_total))
random.shuffle(indices)

train_indices = indices[:n_train]
val_indices = indices[n_train:]

train_recipes = [encoded_recipes[i] for i in train_indices]
val_recipes = [encoded_recipes[i] for i in val_indices]

print(f"학습 세트: {len(train_recipes):,}개 ({len(train_recipes)/n_total*100:.1f}%)")
print(f"검증 세트: {len(val_recipes):,}개 ({len(val_recipes)/n_total*100:.1f}%)")

학습 세트: 181,937개 (90.0%)
검증 세트: 20,216개 (10.0%)


## 4. Dynamic Masking Dataset 생성

In [13]:
# 설정
MAX_LEN = 32  # 최대 시퀀스 길이
MASK_RATIO_RANGE = (0.2, 0.8)  # 동적 마스킹 비율

print(f"최대 시퀀스 길이: {MAX_LEN}")
print(f"마스킹 비율 범위: {MASK_RATIO_RANGE}")

최대 시퀀스 길이: 32
마스킹 비율 범위: (0.2, 0.8)


In [14]:
# 학습 데이터셋 생성
train_dataset = DynamicMaskingDataset(
    recipes=train_recipes,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    mask_ratio_range=MASK_RATIO_RANGE,
    shuffle_ingredients=True,  # 재료 순서 셔플
    bert_style_masking=True    # 80/10/10 마스킹
)

print(f"\n학습 데이터셋 크기: {len(train_dataset):,}")


학습 데이터셋 크기: 181,937


In [15]:
# 검증 데이터셋 생성
val_dataset = DynamicMaskingDataset(
    recipes=val_recipes,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    mask_ratio_range=(0.3, 0.5),  # 검증용 고정 범위
    shuffle_ingredients=False,    # 검증 시 셔플 없음
    bert_style_masking=False      # 단순 마스킹
)

print(f"검증 데이터셋 크기: {len(val_dataset):,}")

검증 데이터셋 크기: 20,216


In [16]:
# 데이터셋 샘플 확인
print("\n=== 학습 데이터 샘플 ===")
sample = train_dataset[0]
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"labels shape: {sample['labels'].shape}")
print(f"masked_positions: {sample['masked_positions']}")

# 디코딩하여 확인
print(f"\n마스킹된 입력: {tokenizer.decode(sample['input_ids'].tolist(), skip_special_tokens=False)}")


=== 학습 데이터 샘플 ===
input_ids shape: torch.Size([32])
attention_mask shape: torch.Size([32])
labels shape: torch.Size([32])
masked_positions: tensor([3])

마스킹된 입력: ['[CLS]', '감자', '소금', '[MASK]', '전분가루', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [17]:
# 상세 샘플 정보
sample_info = train_dataset.get_sample_info(0)
print("\n=== 샘플 상세 정보 ===")
for key, val in sample_info.items():
    print(f"{key}: {val}")


=== 샘플 상세 정보 ===
original_ingredients: ['감자', '소금', '전분가루', '식용유']
num_ingredients: 4
masked_input: ['식용유', '감자', '소금']
num_masked: 1
mask_ratio: 0.25


## 5. DataLoader 테스트

In [18]:
# DataLoader 생성
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Windows 호환
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print(f"학습 배치 수: {len(train_loader):,}")
print(f"검증 배치 수: {len(val_loader):,}")

학습 배치 수: 2,843
검증 배치 수: 316


In [19]:
# 배치 테스트
batch = next(iter(train_loader))
print("\n=== 배치 정보 ===")
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f"{key}: {val.shape}")
    else:
        print(f"{key}: {type(val)}")


=== 배치 정보 ===
input_ids: torch.Size([64, 32])
attention_mask: torch.Size([64, 32])
labels: torch.Size([64, 32])
masked_positions: <class 'list'>


## 6. Leave-One-Out 평가 데이터셋

In [20]:
# Leave-One-Out 평가용 데이터셋 (검증 세트에서)
loo_dataset = LeaveOneOutDataset(
    recipes=val_recipes,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

print(f"Leave-One-Out 평가 샘플 수: {len(loo_dataset):,}")

Leave-One-Out 평가 샘플 수: 157,161


In [21]:
# LOO 샘플 확인
loo_sample = loo_dataset[0]
print("\n=== LOO 샘플 ===")
print(f"input_ids: {loo_sample['input_ids'][:10]}...")
print(f"target_id: {loo_sample['target_id']}")
print(f"masked_position: {loo_sample['masked_position']}")
print(f"target 재료: {tokenizer.get_token(loo_sample['target_id'].item())}")


=== LOO 샘플 ===
input_ids: tensor([  3,   1,  11,  28,  13,  51, 402,  14, 538,   4])...
target_id: 7
masked_position: 1
target 재료: 양파


## 7. 데이터셋 저장

In [22]:
# 데이터셋 저장 (인코딩된 레시피만)
dataset_info = {
    'train_recipes': train_recipes,
    'val_recipes': val_recipes,
    'train_indices': train_indices,
    'val_indices': val_indices,
    'max_len': MAX_LEN,
    'mask_ratio_range': MASK_RATIO_RANGE,
    'vocab_size': tokenizer.vocab_size,
    'version': '3.0.0'
}

# PyTorch 텐서로 저장
torch.save(dataset_info, OUTPUT_DIR / 'dataset_info.pt')
print(f"데이터셋 정보 저장: {OUTPUT_DIR / 'dataset_info.pt'}")

데이터셋 정보 저장: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/dataset_info.pt


In [23]:
# 분할 인덱스 저장 (재현성)
splits = {
    'train_indices': train_indices,
    'val_indices': val_indices,
    'seed': SEED,
    'train_ratio': TRAIN_RATIO
}

with open(OUTPUT_DIR / 'data_splits.json', 'w') as f:
    json.dump(splits, f)
print(f"분할 정보 저장: {OUTPUT_DIR / 'data_splits.json'}")

분할 정보 저장: /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/data_splits.json


In [24]:
# 최종 요약
print("\n" + "="*50)
print("특성 공학 완료")
print("="*50)
print(f"\nTokenizer:")
print(f"  - 어휘 크기: {tokenizer.vocab_size:,}")
print(f"  - 재료 수: {tokenizer.num_ingredients:,}")
print(f"\n데이터셋:")
print(f"  - 학습: {len(train_recipes):,}개 레시피")
print(f"  - 검증: {len(val_recipes):,}개 레시피")
print(f"  - 최대 길이: {MAX_LEN}")
print(f"  - 마스킹 범위: {MASK_RATIO_RANGE}")
print(f"\n출력 파일:")
print(f"  - {OUTPUT_DIR / 'tokenizer.json'}")
print(f"  - {OUTPUT_DIR / 'dataset_info.pt'}")
print(f"  - {OUTPUT_DIR / 'data_splits.json'}")


특성 공학 완료

Tokenizer:
  - 어휘 크기: 12,089
  - 재료 수: 12,084

데이터셋:
  - 학습: 181,937개 레시피
  - 검증: 20,216개 레시피
  - 최대 길이: 32
  - 마스킹 범위: (0.2, 0.8)

출력 파일:
  - /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/tokenizer.json
  - /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/dataset_info.pt
  - /content/drive/MyDrive/Colab Notebooks/SSAFY_Class_18_Team_4_Final_Capstone/data/processed/recipe_v3/data_splits.json
